In [ ]:
import torch
import torch.optim as optim
from itables import init_notebook_mode, show
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import importlib
from ray import tune

import deeparguing.gradual_aacbr as gradual_aacbr
import deeparguing.semantics.relu_semantics as rs
import deeparguing.base_scores.feature_weighted_base_score as fwbs
import deeparguing.casebase_edge_weights.feature_weighted_partial_order as fwpo
import deeparguing.irrelevance_edge_weights.regular_irrelevance as ri

from deeparguing.train import train_model, evaluate_model
from deeparguing.tune import tune_model, objective

from helper import load_iris, split_data, cluster_data



init_notebook_mode(all_interactive=True)

In [ ]:
def reload_imports():
    importlib.reload(gradual_aacbr)
    importlib.reload(rs)
    importlib.reload(fwbs)
    importlib.reload(fwpo)
    importlib.reload(ri)

reload_imports()

In [ ]:
SEED = 42

## Data Set

In [ ]:
X, y = load_iris()
show(X)
show(y)

In [ ]:
all_y = np.unique(y, axis=0)
show(all_y)

## Train Model

### Split into Training, Validation and Test

In [ ]:
train_full, train, val, test = split_data(X, y, SEED)

print(f"Test Size:  {len(test['X'])}")
print(f"Train Size:  {len(train['X'])}")
print(f"Validation Size:  {len(val['X'])}")

### Cluster dataset

In [ ]:
GROUP_PROPORTION = 0.25 
X_centroids, y_centroids = cluster_data(train["X"], train["y"], lambda group_size: int(group_size * GROUP_PROPORTION))

In [ ]:
show(X_centroids)

In [ ]:
X_centroids = torch.tensor(X_centroids, dtype=torch.float32)
y_centroids = torch.tensor(y_centroids, dtype=torch.float32)

### Convert to Torch

In [ ]:
X_train_full, y_train_full = torch.tensor(train_full["X"]), torch.tensor(train_full["y"], dtype=torch.float32)
X_train, y_train = torch.tensor(train["X"]), torch.tensor(train["y"], dtype=torch.float32)
X_val, y_val = torch.tensor(val["X"]), torch.tensor(val["y"], dtype=torch.float32)
X_test, y_test = torch.tensor(test["X"]), torch.tensor(test["y"], dtype=torch.float32)

### Normalize dataset

In [ ]:
train_mean = X_train.mean(dim=0)
train_std = X_train.std(dim=0)

X_train = (X_train - train_mean)/train_std
X_centroids = (X_centroids - train_mean)/train_std
X_val = (X_val - train_mean)/train_std
X_test = (X_test - train_mean)/train_std



In [ ]:
show(X_train.cpu().numpy())

### Train Model

In [ ]:
DEFAULT_CASE = X_train.mean(axis=0)

X_DEFAULTS = DEFAULT_CASE.tile(len(all_y), 1)
Y_DEFAULTS = torch.tensor(all_y)


In [ ]:
MAX_ITERS = 20
EPOCHS = 3000
USE_SYMMETRIC_ATTACKS = False
LR = 0.027724410326258685
MOMENTUM = 0.9
SHARPNESS = 0.6052368105626966
# want it to work regardless of this?
TORCH_SEED = 36




In [ ]:
reload_imports()
torch.manual_seed(TORCH_SEED) # TRY DIFFERENT INITIAL WEIGHTS 

no_features = X_train.shape[-1]
semantics = rs.ReluSemantics(max_iters=MAX_ITERS, epsilon=0)
partial_order = fwpo.FeatureWeightedPartialOrder(no_features, sharpness=SHARPNESS)
irrelevance = ri.RegularIrrelevance(partial_order)
base_score = fwbs.FeatureWeightedBaseScore(no_features)

model = gradual_aacbr.GradualAACBR(semantics, 
                                base_score,
                                irrelevance,
                                partial_order)

criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM)

In [ ]:
reload_imports()
evaluate_model(model, X_centroids, y_centroids, X_DEFAULTS, Y_DEFAULTS, X_val, y_val, show_confusion=True,  print_matrix=True, print_compute_graph=True, print_graph = False )
model.plot_casebase_edge_weights_parameters()
model.plot_base_score_parameters()

In [ ]:
train_model(model, X_centroids, y_centroids, X_train, y_train, X_DEFAULTS, Y_DEFAULTS, optimizer, criterion, EPOCHS, use_symmetric_attacks=False, use_blockers=False, plot_loss_curve=True)

In [ ]:
reload_imports()
with torch.no_grad():
    evaluate_model(model, X_centroids, y_centroids, X_DEFAULTS, Y_DEFAULTS, X_val, y_val, show_confusion=True,  print_matrix=True, print_compute_graph=False, print_graph=False  )

model.plot_casebase_edge_weights_parameters()
model.plot_base_score_parameters()

In [ ]:
assert(False)

### Hyperparameter Tuning

In [ ]:
import random 

seed = random.randint(0, 1000)
torch.manual_seed(seed)
print(seed)

In [ ]:
search_space = {
    "train": train_model,
    "no_iters": tune.choice([15, 20, 25]),
    "epochs": tune.choice([250, 750, 1500, 3000]),
    "lr": tune.loguniform(1e-4, 1e-1),
    "momentum": tune.loguniform(1e-4, 1e-1),
    "sharpness": tune.loguniform(1e-1, 1e1),
    "symmetric_attacks": tune.choice([True, False]),
    "semantics": tune.sample_from(lambda config: rs.ReluSemantics(max_iters=config["no_iters"], epsilon=0)),
    "partial_order": tune.sample_from(lambda config: fwpo.FeatureWeightedPartialOrder(no_features, sharpness=config["sharpness"])),
    "irrelevance": tune.sample_from(lambda config: ri.RegularIrrelevance(config["partial_order"])),
    "base_score": fwbs.FeatureWeightedBaseScore(no_features),
}


In [ ]:
best_results = tune_model(search_space, X_centroids, y_centroids, X_DEFAULTS, Y_DEFAULTS, 
                          X_train, y_train, X_val, y_val, 
                          metric="f1", num_samples=10, disable_tqdm=True)


In [ ]:
print(best_results)


In [ ]:
# best_results = {'no_iters': 20, 'epochs': 3000, 'lr': 0.027724410326258685, 'sharpness': 0.6052368105626966, "momentum": 0.9, 'seed': 36, 'symmetric_attacks': False}

torch.manual_seed(seed)


partial_order = fwpo.FeatureWeightedPartialOrder(no_features, sharpness=best_results["sharpness"])
irrelevance = ri.RegularIrrelevance(partial_order)
base_score = fwbs.FeatureWeightedBaseScore(no_features)

params = {
    "train": train_model,
    "epochs": best_results["epochs"],
    "lr": best_results["lr"],
    "momentum": best_results["momentum"],
    "symmetric_attacks": best_results["symmetric_attacks"],
    "semantics": semantics,
    "partial_order": partial_order,
    "irrelevance": irrelevance,
    "base_score": base_score,
}

# objective(params, X_centroids, y_centroids, X_DEFAULTS, Y_DEFAULTS, X_train, y_train, X_val, y_val, show_confusion=True, plot_loss_curve = True)


objective(params, 
          X_casebase=X_centroids, 
          y_casebase=y_centroids, 
          X_default=X_DEFAULTS, 
          y_default=Y_DEFAULTS, 
          X_train_new_cases=X_train, 
          y_train_new_cases=y_train, 
          X_eval_new_cases=X_val, 
          y_eval_new_cases=y_val, 
          show_confusion=True, 
          plot_loss_curve = True)


### Test Set

In [ ]:
reload_imports()
objective(params, X_centroids, y_centroids, X_DEFAULTS, Y_DEFAULTS, X_train_full, y_train_full, X_test, y_test, show_confusion=True, plot_loss_curve = True)

## Baselines

In [ ]:
from sklearn import neighbors
from sklearn import tree
from sklearn.linear_model import LogisticRegression

y_centroids_label = torch.argmax(y_centroids, dim=1)

logistic_regression = LogisticRegression()
logistic_regression.fit(X_centroids, y_centroids_label)
logistic_predictions = logistic_regression.predict(X_test)
print('Accuracy with Logistic Regression is %.2f' % (accuracy_score(np.argmax(y_test, axis=1), logistic_predictions) * 100))
print('Precision with Logistic Regression is %.2f' % (precision_score(np.argmax(y_test, axis=1), logistic_predictions, average="macro") * 100))
print('Recall with Logistic Regression is %.2f' % (recall_score(np.argmax(y_test, axis=1), logistic_predictions, average="macro") * 100))
print('F1 with Logistic Regression is %.2f' % (f1_score(np.argmax(y_test, axis=1), logistic_predictions, average="macro") * 100))

# %% KNeighborsClassifier
k_classifier = neighbors.KNeighborsClassifier()
k_classifier.fit(X_centroids, y_centroids_label)
k_neighbors_predictions = k_classifier.predict(X_test)
print('Accuracy with KNeighborsClassifier is %.2f' % (accuracy_score(np.argmax(y_test, axis=1), k_neighbors_predictions) * 100))

# %% DecisionTreeClassifier
decision_classifier = tree.DecisionTreeClassifier()
decision_classifier.fit(X_centroids, y_centroids_label)
decision_classifier_predictions = decision_classifier.predict(X_test)
print('Accuracy with DecisionTreeClassifier is %.2f' % (accuracy_score(np.argmax(y_test, axis=1), decision_classifier_predictions) * 100))
